# Лабораторная 1 — повторяемые пробы промптов

Цель: сравнить несколько LLM в задаче нормализации продавца и категоризации расхода по уже извлечённым полям чека и банковской операции. OCR, QR и извлечение суммы здесь **не тестируются**.

Кандидаты: облачный GPT-5 mini, российский GigaChat 2 Lite и локальный Qwen2.5-7B-Instruct через Ollama. Ноутбук не содержит ключа, реальных чеков или выписок. Набор кейсов синтетический и воспроизводимый. Один и тот же набор прогоняется каждым промптом несколько раз, а в результаты записываются ответ, токены, задержка, версия модели и ошибка. Если ячейки с API не запускались, результатов нет — это нужно честно указать в `lab1.md`.

Основание для реализации: Responses API возвращает расход токенов (`usage`) и поддерживает структурированный текстовый ответ; эти поля логируются ниже. [OpenAI API Reference](https://developers.openai.com/api/reference/python/resources/responses/methods/retrieve)

## Протокол

- Модели: включаются только при наличии соответствующей переменной окружения: `OPENAI_API_KEY`, `GIGACHAT_CREDENTIALS`, `LAB1_ENABLE_LOCAL=1`.
- Повторяемость: 6 кейсов × 2 промпта × 3 повтора × число включённых провайдеров. Перед запуском рассчитывается верхняя граница токенов; запуск отменяется при превышении заданного лимита.
- Сравнение: Prompt A — короткая инструкция; Prompt B — фиксированная роль, словарь категорий, правила неопределённости и строгая JSON-схема.
- Метрики: валидность JSON, точность категории, точность нормализации продавца, правильность `needs_review`, средние токены, задержка, оценочная стоимость и доля ошибок API.
- Для Ollama используйте `http://localhost:11434/v1`. Для LM Studio запустите сервер во вкладке Developer и используйте `http://127.0.0.1:1234/v1` — путь `/api/v1` относится к другому, нативному API LM Studio и несовместим с клиентом OpenAI. Базовый URL меняется через `LAB1_LOCAL_BASE_URL`.
- Честное ограничение: шесть синтетических примеров недостаточны для оценки качества на реальных термочеках. После пилота набор нужно расширить обезличенными примерами и вручную размеченными эталонами.

In [17]:
%pip install -q openai gigachat pandas

import json
import os
from datetime import datetime, timezone
from time import perf_counter, sleep
from pathlib import Path

import httpx
import requests

import pandas as pd
from openai import OpenAI
from gigachat import GigaChat
from gigachat.models import ChatCompletionRequest, ChatMessage, ChatModelOptions, ChatResponseFormat

# Ключи задаются в окружении ОС и не сохраняются в notebook.
# PowerShell: $env:OPENAI_API_KEY='...'; $env:GIGACHAT_CREDENTIALS='...'; $env:LAB1_ENABLE_LOCAL='1'
PROVIDERS = {
    'openai': {
        'enabled': bool(os.getenv('OPENAI_API_KEY')), 'model': os.getenv('LAB1_OPENAI_MODEL', 'gpt-5-mini'),
        'input_per_million': 0.25, 'output_per_million': 2.00, 'currency': 'USD',
    },
    'gigachat': {
        'enabled': bool(os.getenv('GIGACHAT_CREDENTIALS')), 'model': os.getenv('LAB1_GIGACHAT_MODEL', 'GigaChat'),
        'input_per_million': 65.0, 'output_per_million': 65.0, 'currency': 'RUB',
    },
    'local_qwen': {
        'enabled': os.getenv('LAB1_ENABLE_LOCAL', '1') == '1', 'model': os.getenv('LAB1_LOCAL_MODEL', 'qwen2.5-7b-instruct-1m'),
        'base_url': os.getenv('LAB1_LOCAL_BASE_URL', 'http://127.0.0.1:1234/v1'),
        'input_per_million': 0.0, 'output_per_million': 0.0, 'currency': 'RUB',
    },
}
ENABLED_PROVIDERS = [name for name, cfg in PROVIDERS.items() if cfg['enabled']]
assert ENABLED_PROVIDERS, 'Включите хотя бы один провайдер через переменную окружения.'

RUNS_PER_CASE = 3
REQUEST_TIMEOUT_SECONDS = int(os.getenv('LAB1_REQUEST_TIMEOUT_SECONDS', '60'))
MAX_INPUT_TOKENS_PER_CALL = 1_000
MAX_OUTPUT_TOKENS_PER_CALL = 150
LOCAL_RETRIES = int(os.getenv('LAB1_LOCAL_RETRIES', '2'))
BUDGET_TOKENS = 50_000  # защитный лимит на весь запуск, не цена
OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)
clients = {}
if 'openai' in ENABLED_PROVIDERS:
    clients['openai'] = OpenAI(timeout=REQUEST_TIMEOUT_SECONDS, max_retries=0)
if 'gigachat' in ENABLED_PROVIDERS:
    clients['gigachat'] = GigaChat(
        base_url='https://api.giga.chat/v1', credentials=os.environ['GIGACHAT_CREDENTIALS'],
        scope=os.getenv('GIGACHAT_SCOPE', 'GIGACHAT_API_PERS'), verify_ssl_certs=False,
    )
if 'local_qwen' in ENABLED_PROVIDERS:
    clients['local_qwen'] = OpenAI(
        base_url=PROVIDERS['local_qwen']['base_url'], api_key='ollama',
        timeout=REQUEST_TIMEOUT_SECONDS, max_retries=0,
    )
# Fail fast: иначе 36 запросов к неработающему локальному серверу выглядят как зависший notebook.
if 'local_qwen' in ENABLED_PROVIDERS:
    local_base_url = PROVIDERS['local_qwen']['base_url'].rstrip('/')
    if local_base_url.endswith('/api/v1'):
        raise ValueError(
            'Для OpenAI-клиента укажите совместимый URL с /v1, а не /api/v1: '
            'например, http://127.0.0.1:1234/v1 для LM Studio.'
        )
    models_url = local_base_url + '/models'
    try:
        health = requests.get(models_url, timeout=3.0)
        health.raise_for_status()
    except httpx.HTTPError as exc:
        raise RuntimeError(
            f'Локальный API недоступен по {models_url}. Запустите Ollama/совместимый сервер '
            f'или отключите local_qwen. Исходная ошибка: {exc}'
        ) from exc
print({'enabled_providers': ENABLED_PROVIDERS, 'runs_per_case': RUNS_PER_CASE, 'request_timeout_seconds': REQUEST_TIMEOUT_SECONDS, 'budget_tokens': BUDGET_TOKENS})

Note: you may need to restart the kernel to use updated packages.
{'enabled_providers': ['local_qwen'], 'runs_per_case': 3, 'request_timeout_seconds': 60, 'budget_tokens': 50000}



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# Боли сегмента: сокращённые банковские описания, разные написания продавца,
# смешанная корзина и операция, по которой нельзя уверенно определить категорию.
# Все данные вымышлены и не содержат персональных данных.
CASES = [
    {
        'id': 'bank_abbreviation',
        'transaction': {'description': 'PYATEROCHKA 123 MOSCOW', 'amount_rub': 1243.50, 'date': '2026-09-01'},
        'receipt': {'merchant': 'ПЯТЁРОЧКА', 'items': ['молоко', 'хлеб', 'яблоки'], 'total_rub': 1243.50},
        'expected': {'merchant_normalized': 'Пятёрочка', 'category': 'Продукты', 'needs_review': False},
    },
    {
        'id': 'taxi_alias',
        'transaction': {'description': 'YANDEX GO MOSKVA', 'amount_rub': 386.00, 'date': '2026-09-02'},
        'receipt': {'merchant': 'Яндекс Go', 'items': ['Поездка'], 'total_rub': 386.00},
        'expected': {'merchant_normalized': 'Яндекс Go', 'category': 'Транспорт', 'needs_review': False},
    },
    {
        'id': 'pharmacy',
        'transaction': {'description': 'APTEKA.RU 8916', 'amount_rub': 712.30, 'date': '2026-09-03'},
        'receipt': {'merchant': 'АПТЕКА.РУ', 'items': ['витамин D', 'пластырь'], 'total_rub': 712.30},
        'expected': {'merchant_normalized': 'Аптека.ру', 'category': 'Здоровье', 'needs_review': False},
    },
    {
        'id': 'marketplace_mixed_basket',
        'transaction': {'description': 'OZON 452010', 'amount_rub': 1549.00, 'date': '2026-09-04'},
        'receipt': {'merchant': 'Ozon', 'items': ['кабель USB', 'тетрадь'], 'total_rub': 1549.00},
        'expected': {'merchant_normalized': 'Ozon', 'category': 'Покупки', 'needs_review': True},
    },
    {
        'id': 'utility_payment',
        'transaction': {'description': 'MOSENERGOSBYT ELEKTRON', 'amount_rub': 2280.14, 'date': '2026-09-05'},
        'receipt': {'merchant': 'Мосэнергосбыт', 'items': ['электроэнергия'], 'total_rub': 2280.14},
        'expected': {'merchant_normalized': 'Мосэнергосбыт', 'category': 'ЖКХ', 'needs_review': False},
    },
    {
        'id': 'ambiguous_sbp_transfer',
        'transaction': {'description': 'СБП ПЕРЕВОД IVAN I.', 'amount_rub': 500.00, 'date': '2026-09-06'},
        'receipt': None,
        'expected': {'merchant_normalized': 'Не определено', 'category': 'Не определено', 'needs_review': True},
    },
]

CATEGORIES = ['Продукты', 'Транспорт', 'Здоровье', 'ЖКХ', 'Покупки', 'Кафе и рестораны', 'Подписки', 'Не определено']
pd.DataFrame([{'id': c['id'], 'bank_description': c['transaction']['description'], 'has_receipt': c['receipt'] is not None} for c in CASES])

,id,bank_description,has_receipt
0,bank_abbreviation,PYATEROCHKA 123 MOSCOW,True
1,taxi_alias,YANDEX GO MOSKVA,True
2,pharmacy,APTEKA.RU 8916,True
3,marketplace_mixed_basket,OZON 452010,True
4,utility_payment,MOSENERGOSBYT ELEKTRON,True
5,ambiguous_sbp_transfer,СБП ПЕРЕВОД IVAN I.,False


In [19]:
PROMPTS = {
    'A_short': '''Нормализуй название продавца и выбери категорию расхода. Ответь JSON с полями merchant_normalized, category, needs_review, reason.''',
    'B_guardrailed': '''Ты помощник учёта расходов. Работай только с переданными полями уже распознанного чека и банковской операции.
Не меняй, не вычисляй и не придумывай сумму, дату или позиции.
Нормализуй продавца в общепринятое название. Выбери ровно одну категорию только из списка.
Если нет чека, данные противоречат друг другу, корзина смешанная или категория не следует из данных — выбери «Не определено» либо наиболее общий вариант и поставь needs_review=true.
reason — одно короткое предложение на русском, только по входным данным.''',
}

JSON_SCHEMA = {
    'format': {
        'type': 'json_schema',
        'name': 'expense_categorization',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'properties': {
                'merchant_normalized': {'type': 'string'},
                'category': {'type': 'string', 'enum': CATEGORIES},
                'needs_review': {'type': 'boolean'},
                'reason': {'type': 'string'},
            },
            'required': ['merchant_normalized', 'category', 'needs_review', 'reason'],
        },
    }
}

def make_input(case):
    return json.dumps(
        {'transaction': case['transaction'], 'receipt': case['receipt'], 'allowed_categories': CATEGORIES},
        ensure_ascii=False,
    )

In [20]:
def estimate_cost(provider, input_tokens, output_tokens):
    cfg = PROVIDERS[provider]
    return (input_tokens or 0) / 1_000_000 * cfg['input_per_million'] + (output_tokens or 0) / 1_000_000 * cfg['output_per_million']

def _openai_usage(row, usage):
    if usage is None:
        return
    row['input_tokens'] = getattr(usage, 'prompt_tokens', None)
    row['output_tokens'] = getattr(usage, 'completion_tokens', None)
    row['total_tokens'] = getattr(usage, 'total_tokens', None)

def _local_completion(instructions, case):
    """Вызов LM Studio с небольшим retry для временного 502.
    Сначала используем json_schema; при 400/422 от старого OpenAI-compatible сервера
    повторяем с json_object. Это совместимее разных версий LM Studio.
    """
    client = clients['local_qwen']
    base_kwargs = {
        'model': PROVIDERS['local_qwen']['model'],
        'messages': [
            {'role': 'system', 'content': instructions},
            {'role': 'user', 'content': make_input(case)},
        ],
        'temperature': 0,
        'max_tokens': MAX_OUTPUT_TOKENS_PER_CALL,
        'stream': False,
    }
    schema_format = {
        'type': 'json_schema',
        'json_schema': {
            'name': 'expense_categorization',
            'strict': True,
            'schema': JSON_SCHEMA['format']['schema'],
        },
    }

    last_exc = None
    for attempt in range(LOCAL_RETRIES + 1):
        try:
            try:
                return client.chat.completions.create(
                    **base_kwargs, response_format=schema_format
                )
            except Exception as exc:
                # Некоторые версии LM Studio/llama.cpp не принимают json_schema.
                status = getattr(exc, 'status_code', None)
                if status in (400, 422):
                    return client.chat.completions.create(
                        **base_kwargs, response_format={'type': 'json_object'}
                    )
                raise
        except Exception as exc:
            last_exc = exc
            status = getattr(exc, 'status_code', None)
            if status == 502 and attempt < LOCAL_RETRIES:
                sleep(1.5 * (attempt + 1))
                continue
            raise last_exc

def run_one(provider, prompt_id, instructions, case, run_number):
    started_at = datetime.now(timezone.utc).isoformat()
    row = {
        'started_at_utc': started_at, 'provider': provider,
        'model_requested': PROVIDERS[provider]['model'], 'prompt_id': prompt_id,
        'case_id': case['id'], 'run_number': run_number, 'raw_output': None, 'error': None,
        'actual_merchant_normalized': None, 'actual_category': None,
        'actual_needs_review': None, 'actual_reason': None,
        'input_tokens': None, 'output_tokens': None, 'total_tokens': None,
    }
    started = perf_counter()
    try:
        if provider == 'gigachat':
            request = ChatCompletionRequest(
                model=PROVIDERS[provider]['model'],
                messages=[
                    ChatMessage(role='system', content=[{'text': instructions}]),
                    ChatMessage(role='user', content=[{'text': make_input(case)}]),
                ],
                model_options=ChatModelOptions(
                    response_format=ChatResponseFormat(
                        type='json_schema',
                        schema=JSON_SCHEMA['format']['schema'],
                        strict=True,
                    )
                ),
            )
            response = clients[provider].chat.create(request)
            row['model_returned'] = response.model
            row['raw_output'] = response.messages[0].content[0].text
            row['input_tokens'] = response.usage.input_tokens
            row['output_tokens'] = response.usage.output_tokens
            row['total_tokens'] = response.usage.total_tokens

        elif provider == 'local_qwen':
            response = _local_completion(instructions, case)
            row['model_returned'] = response.model
            row['raw_output'] = response.choices[0].message.content
            _openai_usage(row, response.usage)

        else:
            request_kwargs = {
                'model': PROVIDERS[provider]['model'],
                'messages': [
                    {'role': 'system', 'content': instructions},
                    {'role': 'user', 'content': make_input(case)},
                ],
                'response_format': {'type': 'json_object'},
            }
            request_kwargs['max_completion_tokens'] = MAX_OUTPUT_TOKENS_PER_CALL
            response = clients[provider].chat.completions.create(**request_kwargs)
            row['model_returned'] = response.model
            row['raw_output'] = response.choices[0].message.content
            _openai_usage(row, response.usage)

        answer = json.loads(row['raw_output'] or '')
        required_keys = {'merchant_normalized', 'category', 'needs_review', 'reason'}
        if not required_keys.issubset(answer):
            row['error'] = f'schema_error: отсутствуют ключи {sorted(required_keys - set(answer))}'
        elif answer.get('category') not in CATEGORIES:
            row['error'] = f'schema_error: недопустимая category={answer.get("category")!r}'
        else:
            row.update({f'actual_{key}': value for key, value in answer.items()})

    except (json.JSONDecodeError, TypeError) as exc:
        row['error'] = f'invalid_json: {exc}'
    except Exception as exc:
        status = getattr(exc, 'status_code', None)
        suffix = f' (HTTP {status})' if status else ''
        row['error'] = f'{type(exc).__name__}{suffix}: {exc}'

    row['latency_ms'] = round((perf_counter() - started) * 1000, 1)
    row['estimated_cost'] = estimate_cost(provider, row['input_tokens'], row['output_tokens'])
    row['cost_currency'] = PROVIDERS[provider]['currency']
    return row

planned_calls = len(ENABLED_PROVIDERS) * len(PROMPTS) * len(CASES) * RUNS_PER_CASE
planned_upper_bound = planned_calls * (MAX_INPUT_TOKENS_PER_CALL + MAX_OUTPUT_TOKENS_PER_CALL)
print(f'Запланировано вызовов: {planned_calls}; верхняя граница: {planned_upper_bound} токенов.')
if planned_upper_bound > BUDGET_TOKENS:
    raise RuntimeError('План превышает BUDGET_TOKENS. Уменьшите RUNS_PER_CASE или набор кейсов до запуска.')

rows = []
completed_calls = 0
try:
    for provider in ENABLED_PROVIDERS:
        for prompt_id, instructions in PROMPTS.items():
            for case in CASES:
                for run_number in range(1, RUNS_PER_CASE + 1):
                    completed_calls += 1
                    print(f'[{completed_calls}/{planned_calls}] {provider} / {prompt_id} / {case["id"]} / повтор {run_number}', flush=True)
                    rows.append(run_one(provider, prompt_id, instructions, case, run_number))
except KeyboardInterrupt:
    print(f'Прогон остановлен пользователем. Сохраняются {len(rows)} завершённых вызовов из {planned_calls}.')

if not rows:
    raise RuntimeError('Нет завершённых вызовов: проверьте ключ, локальный сервер и выбранную модель.')

results = pd.DataFrame(rows)
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
results_path = OUTPUT_DIR / f'lab1_results_{timestamp}.csv'
results.to_csv(results_path, index=False, encoding='utf-8-sig')
print(f'Сохранено: {results_path}')
results[['provider', 'prompt_id', 'case_id', 'run_number', 'actual_category', 'actual_needs_review', 'latency_ms', 'total_tokens', 'estimated_cost', 'cost_currency', 'error']]

Запланировано вызовов: 36; верхняя граница: 41400 токенов.
[1/36] local_qwen / A_short / bank_abbreviation / повтор 1
[2/36] local_qwen / A_short / bank_abbreviation / повтор 2
[3/36] local_qwen / A_short / bank_abbreviation / повтор 3
[4/36] local_qwen / A_short / taxi_alias / повтор 1
[5/36] local_qwen / A_short / taxi_alias / повтор 2
[6/36] local_qwen / A_short / taxi_alias / повтор 3
[7/36] local_qwen / A_short / pharmacy / повтор 1
[8/36] local_qwen / A_short / pharmacy / повтор 2
[9/36] local_qwen / A_short / pharmacy / повтор 3
[10/36] local_qwen / A_short / marketplace_mixed_basket / повтор 1
[11/36] local_qwen / A_short / marketplace_mixed_basket / повтор 2
[12/36] local_qwen / A_short / marketplace_mixed_basket / повтор 3
[13/36] local_qwen / A_short / utility_payment / повтор 1
[14/36] local_qwen / A_short / utility_payment / повтор 2
[15/36] local_qwen / A_short / utility_payment / повтор 3
[16/36] local_qwen / A_short / ambiguous_sbp_transfer / повтор 1
[17/36] local_qwen

,provider,prompt_id,case_id,run_number,actual_category,actual_needs_review,latency_ms,total_tokens,estimated_cost,cost_currency,error
0,local_qwen,A_short,bank_abbreviation,1,Продукты,False,1045.3,227,0.0,RUB,None
1,local_qwen,A_short,bank_abbreviation,2,Продукты,False,573.2,227,0.0,RUB,None
2,local_qwen,A_short,bank_abbreviation,3,Продукты,False,562.9,227,0.0,RUB,None
3,local_qwen,A_short,taxi_alias,1,Транспорт,False,566.5,205,0.0,RUB,None
4,local_qwen,A_short,taxi_alias,2,Транспорт,False,532.5,205,0.0,RUB,None
5,local_qwen,A_short,taxi_alias,3,Транспорт,False,539.4,205,0.0,RUB,None
6,local_qwen,A_short,pharmacy,1,Здоровье,False,599.7,223,0.0,RUB,None
7,local_qwen,A_short,pharmacy,2,Здоровье,False,570.7,223,0.0,RUB,None
8,local_qwen,A_short,pharmacy,3,Здоровье,False,567.0,223,0.0,RUB,None
9,local_qwen,A_short,marketplace_mixed_basket,1,Покупки,False,554.2,215,0.0,RUB,None


In [21]:
# Оценка выполняется только по заранее заданным эталонам; ответы не исправляются вручную.
expected = pd.DataFrame([{'case_id': c['id'], **c['expected']} for c in CASES])
scored = results.merge(expected, on='case_id', how='left')
scored['json_valid'] = scored['error'].isna() & scored['actual_category'].notna()
scored['category_correct'] = scored['actual_category'].eq(scored['category'])
scored['merchant_correct'] = scored['actual_merchant_normalized'].str.casefold().eq(scored['merchant_normalized'].str.casefold())
scored['review_correct'] = scored['actual_needs_review'].eq(scored['needs_review'])

summary = (
    scored.groupby(['provider', 'model_requested', 'prompt_id'], dropna=False)
    .agg(
        calls=('case_id', 'size'),
        json_valid_rate=('json_valid', 'mean'),
        category_accuracy=('category_correct', 'mean'),
        merchant_accuracy=('merchant_correct', 'mean'),
        review_accuracy=('review_correct', 'mean'),
        api_error_rate=('error', lambda s: s.notna().mean()),
        mean_total_tokens=('total_tokens', 'mean'),
        mean_latency_ms=('latency_ms', 'mean'),
        total_estimated_cost=('estimated_cost', 'sum'),
    )
    .reset_index()
)
display(summary)
display(scored[['provider', 'prompt_id', 'case_id', 'actual_merchant_normalized', 'actual_category', 'actual_needs_review', 'latency_ms', 'estimated_cost', 'cost_currency', 'error']])
summary.to_csv(OUTPUT_DIR / f'lab1_summary_{timestamp}.csv', index=False, encoding='utf-8-sig')

,provider,model_requested,prompt_id,calls,json_valid_rate,category_accuracy,merchant_accuracy,review_accuracy,api_error_rate,mean_total_tokens,mean_latency_ms,total_estimated_cost
0,local_qwen,qwen2.5-7b-instruct-1m,A_short,18,1.0,1.000000,0.666667,0.666667,0.0,214.666667,602.627778,0.0
1,local_qwen,qwen2.5-7b-instruct-1m,B_guardrailed,18,1.0,0.833333,0.666667,0.833333,0.0,350.833333,671.783333,0.0


,provider,prompt_id,case_id,actual_merchant_normalized,actual_category,actual_needs_review,latency_ms,estimated_cost,cost_currency,error
0,local_qwen,A_short,bank_abbreviation,Пяточка,Продукты,False,1045.3,0.0,RUB,None
1,local_qwen,A_short,bank_abbreviation,Пяточка,Продукты,False,573.2,0.0,RUB,None
2,local_qwen,A_short,bank_abbreviation,Пяточка,Продукты,False,562.9,0.0,RUB,None
3,local_qwen,A_short,taxi_alias,Яндекс Go,Транспорт,False,566.5,0.0,RUB,None
4,local_qwen,A_short,taxi_alias,Яндекс Go,Транспорт,False,532.5,0.0,RUB,None
5,local_qwen,A_short,taxi_alias,Яндекс Go,Транспорт,False,539.4,0.0,RUB,None
6,local_qwen,A_short,pharmacy,Аптека.РУ,Здоровье,False,599.7,0.0,RUB,None
7,local_qwen,A_short,pharmacy,Аптека.Ру,Здоровье,False,570.7,0.0,RUB,None
8,local_qwen,A_short,pharmacy,Аптека.Ру,Здоровье,False,567.0,0.0,RUB,None
9,local_qwen,A_short,marketplace_mixed_basket,Ozon,Покупки,False,554.2,0.0,RUB,None


In [22]:
# Этот черновик создаётся ПОСЛЕ прогона. Он не утверждает, что эксперимент был проведён,
# пока в папке results нет CSV с фактическими ответами.
best = summary.sort_values(['category_accuracy', 'json_valid_rate', 'api_error_rate'], ascending=[False, False, True]).iloc[0]
failed = scored.loc[~scored['category_correct'] | ~scored['json_valid'], ['provider', 'prompt_id', 'case_id', 'raw_output', 'error']]

draft = f'''# Лабораторная работа 1: пробы промптов

## Что проверяли
Проверяли нормализацию продавца и категоризацию расходов по синтетическим, обезличенным данным. OCR, QR и извлечение сумм не оценивались. Провайдеры: `{', '.join(ENABLED_PROVIDERS)}`. Повторов на кейс: {RUNS_PER_CASE}. Всего API-вызовов: {len(results)}.

## Что пробовали
- **A_short**: краткая инструкция без правил неопределённости.
- **B_guardrailed**: фиксированные категории, запрет придумывать финансовые поля, признак ручной проверки и строгая JSON-схема.

## Что вышло
{summary.to_markdown(index=False)}

## Что не вышло / ограничения
Ошибочные или невалидные ответы (показываются без исправления):
{failed.to_markdown(index=False) if not failed.empty else 'Не обнаружены в данном малом наборе; это не доказывает отсутствие ошибок на реальных данных.'}

Набор состоит из шести синтетических кейсов и не измеряет точность OCR, качество QR-данных, задержку при нагрузке или поведение на реальных персональных данных.

## Черновой вывод
Лучший по заданным метрикам вариант: **{best['provider']} / {best['model_requested']} / {best['prompt_id']}** (точность категории: {best['category_accuracy']:.0%}, валидный JSON: {best['json_valid_rate']:.0%}, средняя задержка: {best['mean_latency_ms']:.0f} мс, ошибок API: {best['api_error_rate']:.0%}). Этот вывод предварительный: он опирается только на сохранённые прогоны и малый синтетический набор. Рабочим для следующего шага выглядит подход с фиксированной схемой ответа, ограниченным словарём категорий и обязательным `needs_review` для неоднозначных операций.
'''
draft_path = OUTPUT_DIR / f'lab1_draft_{timestamp}.md'
draft_path.write_text(draft, encoding='utf-8')
print(f'Черновик для переноса в lab1.md: {draft_path}')
print(draft)

Черновик для переноса в lab1.md: results\lab1_draft_20260923T172128Z.md
# Лабораторная работа 1: пробы промптов

## Что проверяли
Проверяли нормализацию продавца и категоризацию расходов по синтетическим, обезличенным данным. OCR, QR и извлечение сумм не оценивались. Провайдеры: `local_qwen`. Повторов на кейс: 3. Всего API-вызовов: 36.

## Что пробовали
- **A_short**: краткая инструкция без правил неопределённости.
- **B_guardrailed**: фиксированные категории, запрет придумывать финансовые поля, признак ручной проверки и строгая JSON-схема.

## Что вышло
| provider   | model_requested        | prompt_id     |   calls |   json_valid_rate |   category_accuracy |   merchant_accuracy |   review_accuracy |   api_error_rate |   mean_total_tokens |   mean_latency_ms |   total_estimated_cost |
|:-----------|:-----------------------|:--------------|--------:|------------------:|--------------------:|--------------------:|------------------:|-----------------:|--------------------:|-------------

## Как интерпретировать результат

Переносить в `lab1.md` нужно фактический созданный файл `results/lab1_draft_*.md`, включая ошибки. Не следует писать «модель справилась», если API-прогон не был выполнен или если в таблице есть невалидные ответы.

Критерий перехода к следующему этапу: на расширенном обезличенном наборе не менее 100 кейсов выбранный промпт даёт валидный JSON ≥ 98%, точность категории ≥ 85% и не помечает однозначный кейс как автоматически подтверждённый при ожидаемом `needs_review=true`. Иначе остаётся только rule-based категоризация с ручным выбором.